# Run the next two cells

In [115]:
# --- In your notebook ---
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import pandas as pd
from itertools import islice
from IPython.display import display
import re


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [109]:

repo_root = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# sanity checks
assert (repo_root / "py_files").is_dir(), "py_files/ folder not found"
assert (repo_root / "py_files" / "export_slsne_photometry.py").is_file(), "module file not found"

from py_files.export_slsne_photometry import (
    read_supernova_table_txt, to_export,
    process_one, process_all_events, load_allparams_robust, print_cols, resolve_cols 
)

supernovae_dir = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/SLSNe/slsne/ref_data/supernovae")
out_perevent   = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/output/per_event_files")
out_allevent   = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/output/all_events")


# Parameter Control, Test directory locations, and presence of specific event can be found. 

In [56]:
# Output format controls
write_parquet = True   # set False if you don't want parquet
write_csv = True   # CSV mirror for auditing

# Quick sanity checks
print("Repo root:        ", repo_root)
print("Supernovae dir:   ", supernovae_dir, "exists:", supernovae_dir.exists())
print("Output directory: ", out_perevent, "exists:", out_perevent.exists())


# quick smoke test (replace with an event folder that exists in your clone)
test_event = "2005ap"
print((supernovae_dir / test_event / f"{test_event}.txt").exists())


Repo root:         /Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric
Supernovae dir:    /Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/SLSNe/slsne/ref_data/supernovae exists: True
Output directory:  /Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/output/per_event_files exists: True
True


## Process an indivudal event .txt file into a table and see if it has been created. This will not create the csv. 

In [62]:
# IMPORTANT: pass the directory for the event, not the string's .parent
df_one = process_one(supernovae_dir / "2005ap", "2005ap",
                     out_perevent, write_parquet=True, write_csv=True)
display(df_one.head(10))

# Test on 2005ap


print("Rows:", len(df_one))
print("detected value counts:\n", df_one["detected"].value_counts(dropna=False))
print("CSV exists? ", (out_perevent / "2005ap.csv").exists())
print("Parquet exists? ", (out_perevent / "2005ap.parquet").exists())


,mjd,mag,mag_err,UL,filter,detected
0,53377.4,19.618,inf,True,r,0
1,53379.4,18.938,inf,True,r,0
2,53384.4,19.228,inf,True,r,0
3,53386.5,19.438,inf,True,r,0
4,53387.5,19.138,inf,True,r,0
5,53389.5,18.948,inf,True,r,0
6,53408.4,19.688,inf,True,r,0
7,53415.4,18.648,inf,True,r,0
8,53430.2,18.628,inf,True,r,0
9,53432.3,18.498,0.11,False,r,1


Rows: 50
detected value counts:
 detected
1    32
0    18
Name: count, dtype: int64
CSV exists?  True
Parquet exists?  True


## Read original txt file in a table format, display it, Export the modified table to csv/parquet, display it

In [65]:

# Adjust this to your local path
file_2005ap = supernovae_dir / "2005ap" / "2005ap.txt"

raw = read_supernova_table_txt(file_2005ap)
print("Columns:", list(raw.columns))     # should include MJD, Mag, MagErr, Filter, UL, etc.
display(raw.head(5))

df = to_export(raw)
display(df.head(30))

# Quick checks
print("NaNs per column:\n", df.isna().sum())
print("detected counts:\n", df["detected"].value_counts(dropna=False))
print("unique filters:", df["filter"].unique())


Columns: ['MJD', 'Mag', 'Raw', 'MagErr', 'Telescope', 'Instrument', 'Filter', 'UL', 'System', 'Ignore', 'Source']


,MJD,Mag,Raw,MagErr,Telescope,Instrument,Filter,UL,System,Ignore,Source
0,53377.4,19.618,19.64,-1.0,ROTSE-III,--,R,True,Vega,True,2007ApJ...668L..99Q
1,53379.4,18.938,18.96,-1.0,ROTSE-III,--,R,True,Vega,True,2007ApJ...668L..99Q
2,53384.4,19.228,19.25,-1.0,ROTSE-III,--,R,True,Vega,True,2007ApJ...668L..99Q
3,53386.5,19.438,19.46,-1.0,ROTSE-III,--,R,True,Vega,True,2007ApJ...668L..99Q
4,53387.5,19.138,19.16,-1.0,ROTSE-III,--,R,True,Vega,True,2007ApJ...668L..99Q


,mjd,mag,mag_err,UL,filter,detected
0,53377.4000,19.6180,inf,True,r,0
1,53379.4000,18.9380,inf,True,r,0
2,53384.4000,19.2280,inf,True,r,0
3,53386.5000,19.4380,inf,True,r,0
4,53387.5000,19.1380,inf,True,r,0
5,53389.5000,18.9480,inf,True,r,0
6,53408.4000,19.6880,inf,True,r,0
7,53415.4000,18.6480,inf,True,r,0
8,53430.2000,18.6280,inf,True,r,0
9,53432.3000,18.4980,0.1100,False,r,1


NaNs per column:
 mjd         0
mag         0
mag_err     0
UL          0
filter      0
detected    0
dtype: int64
detected counts:
 detected
1    32
0    18
Name: count, dtype: int64
unique filters: <StringArray>
['r', 'g', 'u', 'i']
Length: 4, dtype: string


## Process all events in /supernovae folder, put into csv and parquet for each event, put all events into one single csv and parquet 

In [69]:

index_df, combined_csv, combined_parq = process_all_events(
    supernovae_dir, out_perevent, out_allevent,
    include_ul=True, make_combined=True,
    write_parquet=True, write_csv=True
)
display(index_df.head(15))
print("Index:", out_perevent / "_index.csv")
print("Combined CSV:", combined_csv)
print("Combined Parquet:", combined_parq)

Exporting SLSNe per-object:   0%|          | 0/265 [00:00<?, ?it/s]

,event,status,path,n_rows,n_detected,n_limits
0,1991D,ok,/Users/andradenebula/Documents/Research/Transi...,19,17,2
1,1999as,ok,/Users/andradenebula/Documents/Research/Transi...,43,43,0
2,1999bz,ok,/Users/andradenebula/Documents/Research/Transi...,1,1,0
3,2002gh,ok,/Users/andradenebula/Documents/Research/Transi...,45,45,0
4,2005ap,ok,/Users/andradenebula/Documents/Research/Transi...,50,32,18
5,2006oz,ok,/Users/andradenebula/Documents/Research/Transi...,95,70,25
6,2007bi,ok,/Users/andradenebula/Documents/Research/Transi...,138,133,5
7,2009cb,ok,/Users/andradenebula/Documents/Research/Transi...,31,31,0
8,2009jh,ok,/Users/andradenebula/Documents/Research/Transi...,22,22,0
9,2010gx,ok,/Users/andradenebula/Documents/Research/Transi...,273,260,13


Index: /Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/output/per_event_files/_index.csv
Combined CSV: /Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/output/all_events/all_objects.csv
Combined Parquet: /Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/output/all_events/all_objects.parquet


# All Parameter Table 

##  Pull the all parameter table .txt, format it to see the first few lines, then print all the column names with indices

In [92]:
# Build the path (yours already does this)
allparams_path = repo_root / "SLSNe" / "slsne" / "ref_data" / "all_parameters.txt"
print("all_parameters path:", allparams_path)
assert allparams_path.exists(), "Could not find all_parameters.txt at the expected path."

# 1) Load the table
from py_files.export_slsne_photometry import load_allparams_robust
allparams_df = load_allparams_robust(allparams_path)

# 2) Print first few rows (optional)
from IPython.display import display
display(allparams_df.head())

# 3) Print column names with indices
for i, c in enumerate(allparams_df.columns):
    print(f"{i:>3}: {c}")


all_parameters path: /Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/SLSNe/slsne/ref_data/all_parameters.txt


,name,redshift_lo,redshift_med,redshift_up,texplosion_lo,texplosion_med,texplosion_up,fnickel_lo,fnickel_med,fnickel_up,...,r_peak_up,frac_lo,frac_med,frac_up,1frac_lo,1frac_med,1frac_up,efficiency_lo,efficiency_med,efficiency_up
0,1991D,0.0,0.04179,0.0,8.85557,-36.74645,6.08271,0.02723,0.03564,0.09394,...,0.36293,0.04075,0.99688,0.00303,0.00303,0.00312,0.04075,0.06205,0.11002,0.1464
1,1999as,0.0,0.127,0.0,9.61176,-32.81598,5.35865,0.00246,0.00389,0.00564,...,0.81024,0.00034,0.9999,8e-05,8e-05,0.0001,0.00034,0.03887,0.07776,0.08389
2,1999bz,0.0,0.0846,0.0,38.45281,-142.03278,50.37476,0.0918,0.336,0.11103,...,0.21678,0.03178,0.03205,0.49241,0.49241,0.96795,0.03178,0.38871,0.85337,0.6155
3,2002gh,0.0,0.3653,0.0,7.07357,-89.59422,9.65322,0.01792,0.02058,0.08967,...,0.07914,0.02044,0.99256,0.00642,0.00642,0.00744,0.02044,0.33218,0.91064,0.56285
4,2005ap,0.0,0.2832,0.0,4.27156,-8.92517,2.45517,0.01018,0.01341,0.04841,...,0.24393,0.01106,0.99808,0.00179,0.00179,0.00192,0.01106,0.1707,0.26925,0.32174


  0: name
  1: redshift_lo
  2: redshift_med
  3: redshift_up
  4: texplosion_lo
  5: texplosion_med
  6: texplosion_up
  7: fnickel_lo
  8: fnickel_med
  9: fnickel_up
 10: Pspin_lo
 11: Pspin_med
 12: Pspin_up
 13: log(Bfield)_lo
 14: log(Bfield)_med
 15: log(Bfield)_up
 16: Mns_lo
 17: Mns_med
 18: Mns_up
 19: thetaPB_lo
 20: thetaPB_med
 21: thetaPB_up
 22: mejecta_lo
 23: mejecta_med
 24: mejecta_up
 25: kappa_lo
 26: kappa_med
 27: kappa_up
 28: kappagamma_lo
 29: kappagamma_med
 30: kappagamma_up
 31: vejecta_lo
 32: vejecta_med
 33: vejecta_up
 34: temperature_lo
 35: temperature_med
 36: temperature_up
 37: alpha_lo
 38: alpha_med
 39: alpha_up
 40: cutoff_wavelength_lo
 41: cutoff_wavelength_med
 42: cutoff_wavelength_up
 43: log(nhhost)_lo
 44: log(nhhost)_med
 45: log(nhhost)_up
 46: A_V_lo
 47: A_V_med
 48: A_V_up
 49: MJD0_lo
 50: MJD0_med
 51: MJD0_up
 52: log(kenergy)_lo
 53: log(kenergy)_med
 54: log(kenergy)_up
 55: mnickel_lo
 56: mnickel_med
 57: mnickel_up
 58: log

##  Load all param table, preview first column, select columns you want to include besides the anchor, resolve indices to acutal names from list, build filename from selected columns, write to csv and parquet. Pull redshift or any other parameter from Sebastian's all parameter table and make it into a csv/parquet 

In [117]:
# --- load the table ---
allparams_df = load_allparams_robust(allparams_path)

# Preview just the anchor column (first col or 'name' if present)
print_cols(allparams_df, None, head=10)

# Choose columns by index (0-based). Example: pick column #2 besides the anchor.
selected_cols = [2]

# Resolve to actual names (prepend anchor)
resolved = resolve_cols(allparams_df, selected_cols)
df_cols = allparams_df[resolved].copy()

# Coerce numerics for non-anchor columns
for c in resolved[1:]:
    df_cols[c] = pd.to_numeric(df_cols[c], errors="coerce")

# Build filename stub from the *names* for your selected indices (excluding anchor)
idx_names = [allparams_df.columns[i] for i in selected_cols if 0 <= i < len(allparams_df.columns)]
sanitize = lambda s: re.sub(r"[^A-Za-z0-9._-]+", "_", s)
stub = "_".join(sanitize(c) for c in idx_names) if idx_names else "cols"

# Write outputs
if write_parquet:
    df_cols.to_parquet(out_allevent / f"allevent_{stub}.parquet",
                       index=False, engine="pyarrow", compression="zstd", compression_level=7)
if write_csv:
    df_cols.to_csv(out_allevent / f"allevent_{stub}.csv", index=False)

display(df_cols.head(10))
print("Columns used (resolved):", resolved)
print("Filename stub:", stub)

,name
0,1991D
1,1999as
2,1999bz
3,2002gh
4,2005ap
5,2006oz
6,2007bi
7,2009cb
8,2009jh
9,2010gx


,name,redshift_med
0,1991D,0.04179
1,1999as,0.12700
2,1999bz,0.08460
3,2002gh,0.36530
4,2005ap,0.28320
5,2006oz,0.37600
6,2007bi,0.12790
7,2009cb,0.18670
8,2009jh,0.34990
9,2010gx,0.22970


Columns used (resolved): ['name', 'redshift_med']
Filename stub: redshift_med
